In [1]:
!uv sync

Resolved 53 packages in 4ms
Audited 47 packages in 204ms


In [24]:
import requests
import os
import pandas as pd

from streampipes.client import StreamPipesClient
from streampipes.client.config import StreamPipesClientConfig
from streampipes.client.credential_provider import StreamPipesApiKeyCredentials

from dotenv import load_dotenv
from enum import Enum

load_dotenv()

api_user = os.getenv("SP_API_USER")
api_token = os.getenv("SP_API_TOKEN")
host = os.getenv("SP_API_HOST")
port = os.getenv("SP_API_PORT", 80)

missing = [
    name
    for name, value in {"SP_API_USER": api_user, "SP_API_TOKEN": api_token, "SP_API_HOST": host, "SP_API_PORT": port}.items()
    if not value
]
if missing:
    raise ValueError(f"Missing required environment variable(s): {', '.join(missing)}")

In [16]:
config = StreamPipesClientConfig(
    credential_provider=StreamPipesApiKeyCredentials(
		username= api_user,
		api_key= api_token
	),
    host_address=host,
	port=port,
	https_disabled=True
)

## Enums

In [17]:
class ContentType(Enum):
	JSON = "application/json"
	YAML = "application/x-yaml"
	YML = "application/yml"
	ZIP = "application/zip"
	ALL = "*/*"

class ApplicationMethod(Enum):
	GET = "GET"
	POST = "POST"
	DELETE = "DELETE"
	PUT = "PUT"

class AssetEndpoints(Enum):
	FETCH_ALL = "v2/assets"
	FETCH_BY_ID = "v2/assets/{treeId}"
	CREATE = "v2/assets/{treeId}"
	UPDATE = "v2/assets/{treeId}"
	DELETE = "v2/assets/{treeId}/{revId}"

class AdapterEndpoints(Enum):
	FETCH_ALL = "v2/connect/master/adapters"
	FETCH_BY_ID = "v2/connect/master/adapters/{adapterId}"
	DELETE = "v2/connect/master/adapters/{adapterId}"

class CompactAdapterEndpoints(Enum):
	CREATE_COMPACT = "v2/connect/compact-adapters"

class PipelineTemplateEndpoints(Enum):
	FETCH_ALL = "v2/pipeline-templates"

class DescriptionEndpoints(Enum):
	FETCH_ADAPTER_BUNDLES_ASSETS = "v2/connect/master/description/{id}/assets"
	FETCH_ADAPTER_TYPES = "v2/connect/master/description/adapters"

class LocationEndpoints(Enum):
	FETCH_ALL = "v2/admin/location-config"

class AppId(Enum):
	OPCUA = "org.apache.streampipes.connect.iiot.adapters.opcua"

## Api Request Handler

In [18]:
class StreamPipesAPI():
	def __init__(self, config: StreamPipesClientConfig, host: str, disable_https: bool = False) -> None:
		self.client = config
		self.base_url = f"{'http://' if disable_https else 'https://'}" \
						f"{host}:81" \
						f"/streampipes-backend/api"

	def call(
		self, 
		method: ApplicationMethod, 
		endpoint: str, 
		c_type: ContentType | None = None, 
		payload: str | bytes | None = None
	) -> dict | None:
		return self._request(
			method=method,
			endpoint=endpoint,
			c_type=c_type,
			payload=payload
		)

	# ----------------------------
	# Core HTTP request helper
	# ----------------------------

	def _request(
			self, 
			method: ApplicationMethod, 
			endpoint: str | None = None, 
			c_type: ContentType | None = None,
			payload: str | bytes | None = None, 
		) -> dict | None:
		normalized_base = self.base_url.rstrip("/")
		normalized_endpoint = endpoint.strip().strip("/") if endpoint else ""
		url = f"{normalized_base}/{normalized_endpoint}" if normalized_endpoint else normalized_base

		c_type = c_type or getattr(self, "c_type", None)
		if c_type is None:
			raise ValueError("Content type must be set either in the method call or previously in the instance.")

		headers = {
			**self.client.credential_provider.make_headers(),
			"Content-Type": (c_type or ContentType.YML).value, #c_type.value
			"Accept": ContentType.JSON.value, #c_type.value
		}

		data = None
		json_data = None
		if payload is not None:
			if isinstance(payload, (str, bytes)):
				data = payload
			elif isinstance(payload, dict):
				json_data = payload

		# just before the response = getattr(...) call
		if data is not None:
			print(f"Sending raw data ({len(data)} bytes):")
			print(data.decode("utf-8") if isinstance(data, bytes) else data)
		elif json_data is not None:
			print(f"Sending JSON: {json_data}")
		else:
			print("No payload being sent!")   # ← if you see this, bytes fell through

		response = None
		try:
			response = getattr(requests, method.value.lower())(
				url, 
				headers=headers,
				data=data,
				json=json_data,
				timeout=10
			)
			response.raise_for_status()
			if c_type == ContentType.ZIP:
				print(f"[{method.value}] {url} → {response.status_code} (ZIP content)")
				return response.content
			if response.text:
				print(f"[{method.value}] {url} → {response.status_code}")
				return response.json()
			else:
				print(f"[{method.value}] {url} → {response.status_code} (no content)")
				return None
		except requests.RequestException as e:
			print(f"[{method.value}] {url} failed: {e}")
			if response is not None:
				print("Status:", response.status_code)
				print("Response body:", response.text)
			raise

In [19]:
api = StreamPipesAPI(config=config, host=host, disable_https=True)

## Requests

In [36]:
pipelines = api.call(
	method=ApplicationMethod.GET,
	endpoint="v2/pipelines",
	c_type=ContentType.JSON
)

adapters = api.call(
	method=ApplicationMethod.GET,
	endpoint="v2/connect/master/adapters",
	c_type=ContentType.JSON
)

assets = api.call(
	method=ApplicationMethod.GET,
	endpoint="v2/assets",
	c_type=ContentType.JSON
)

No payload being sent!
[GET] http://10.249.127.101:81/streampipes-backend/api/v2/pipelines → 200
No payload being sent!
[GET] http://10.249.127.101:81/streampipes-backend/api/v2/connect/master/adapters → 200
No payload being sent!
[GET] http://10.249.127.101:81/streampipes-backend/api/v2/assets → 200


## api Responses

In [26]:
pd.DataFrame(pipelines)

,_id,_rev,actions,createdAt,createdByUser,description,elementId,healthStatus,labels,lastMigratedAt,name,pipelineNotifications,publicElement,restartOnSystemReboot,running,sepas,startedAt,streams,valid
0,0647b23a32ae4d5e8b162a538865f7de,2-573805acfa9aa6f8bf33975b156fe3b4,[{'@class': 'org.apache.streampipes.model.grap...,1781696677110,dd787510f8e240699c0eee8496d819f1,MQTT publish: PLC641_LP6ECP92PQF2,0647b23a32ae4d5e8b162a538865f7de,OK,[],0,mqtt-PLC641_LP6ECP92PQF2,[],False,False,True,[],1781696677198,[{'@class': 'org.apache.streampipes.model.SpDa...,True
1,96c607b04d36456796ca7ab2177dc8cb,2-f07d81bd20ce0fa2872c891503fbdf8e,[{'@class': 'org.apache.streampipes.model.grap...,1781696566836,dd787510f8e240699c0eee8496d819f1,MQTT publish: PLC641_PR6ECP41M1,96c607b04d36456796ca7ab2177dc8cb,OK,[],0,mqtt-PLC641_PR6ECP41M1,[],False,False,True,[],1781696566922,[{'@class': 'org.apache.streampipes.model.SpDa...,True


In [35]:
pd.DataFrame(adapters)

,@class,appId,config,connectedTo,correspondingDataStreamElementId,correspondingServiceGroup,createdAt,dataStream,deploymentConfiguration,description,...,includesLocales,internallyManaged,name,rev,rules,running,selectedEndpointUrl,selectedServiceId,transformationConfig,version
0,org.apache.streampipes.model.connect.adapter.A...,org.apache.streampipes.connect.iiot.adapters.p...,[{'@class': 'org.apache.streampipes.model.stat...,None,sp:spdatastream:qDNVYa,None,1781696676889,{'@class': 'org.apache.streampipes.model.SpDat...,"{'desiredServiceTags': [], 'selectedEndpointUr...",ktl rinsing and ventilation,...,True,False,PLC641_LP6ECP92PQF2,3-ada565b5ab473f40c7b2d5d9885d85e1,[],True,http://172.31.0.2:8090,org.apache.streampipes.extensions.all.iiot-EVhi5I,"{'inputs': [], 'language': 'javascript', 'outp...",1
1,org.apache.streampipes.model.connect.adapter.A...,org.apache.streampipes.connect.iiot.adapters.p...,[{'@class': 'org.apache.streampipes.model.stat...,None,sp:spdatastream:OSHOxj,None,1781696566676,{'@class': 'org.apache.streampipes.model.SpDat...,"{'desiredServiceTags': [], 'selectedEndpointUr...",,...,True,False,PLC641_PR6ECP41M1,3-11f4b7aa3bb0ee16e3d196e6eb11e05d,[],True,http://172.31.0.2:8090,org.apache.streampipes.extensions.all.iiot-EVhi5I,"{'inputs': [], 'language': 'javascript', 'outp...",1


In [37]:
pd.DataFrame(assets)

,additionalData,appDocType,assetDescription,assetId,assetLinks,assetName,assetSite,assetType,assets,elementId,labelIds,removable,rev
0,"{'maximo_route': 'B', 'maximo_loc_hierarchy_id...",asset-management,paintshop,None,[],B,"{'siteId': None, 'area': None, 'hasExactLocati...","{'assetIcon': None, 'assetIconColor': None, 'a...","[{'additionalData': {'maximo_route': 'B; B-1',...",sp:spassetmodel:TzLYKu,[],False,504-4d247a72b7577acb7e433474c308d5ec
